In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

In [6]:
# Create synthetic dataset
n_samples = 1000

times = pd.date_range("2020-01-01", periods=n_samples, freq="D")
labels = pd.DataFrame({
    "t0": times,
    "t1": times + pd.to_timedelta(np.random.randint(1, 5, n_samples), unit="D"),
    "feature": np.random.randn(n_samples),
    "label": np.random.choice([0, 1], size=n_samples)
})

labels.head()


,t0,t1,feature,label
0,2020-01-01,2020-01-02,-0.611518,1
1,2020-01-02,2020-01-05,-1.406661,1
2,2020-01-03,2020-01-04,-0.923233,0
3,2020-01-04,2020-01-08,-1.351685,1
4,2020-01-05,2020-01-07,-0.975873,0


In [8]:
# Standard 10-fold cross-validation
kf = KFold(n_splits=10, shuffle=False)

for train_idx, test_idx in kf.split(labels):
    print("Train:", train_idx[:5], "Test:", test_idx[:5])


Train: [100 101 102 103 104] Test: [0 1 2 3 4]
Train: [0 1 2 3 4] Test: [100 101 102 103 104]
Train: [0 1 2 3 4] Test: [200 201 202 203 204]
Train: [0 1 2 3 4] Test: [300 301 302 303 304]
Train: [0 1 2 3 4] Test: [400 401 402 403 404]
Train: [0 1 2 3 4] Test: [500 501 502 503 504]
Train: [0 1 2 3 4] Test: [600 601 602 603 604]
Train: [0 1 2 3 4] Test: [700 701 702 703 704]
Train: [0 1 2 3 4] Test: [800 801 802 803 804]
Train: [0 1 2 3 4] Test: [900 901 902 903 904]


In [10]:
# Purged  cross-validation, If a test event overlaps with training events, those training samples are purged.

def purged_cv_split(events, n_splits=10, embargo=0.01):

    #embargo: fraction of data to embargo after each test fold
    
    n_samples = len(events)
    embargo_size = int(n_samples * embargo)
    
    kf = KFold(n_splits=n_splits, shuffle=False)
    
    for train_idx, test_idx in kf.split(events):
        test_start, test_end = test_idx[0], test_idx[-1]
        
        # Apply embargo
        test_end_embargo = min(test_end + embargo_size, n_samples)
        
        # Drop overlapping training samples
        t0_test = events.iloc[test_idx].t0.min()
        t1_test = events.iloc[test_idx].t1.max()
        
        train_idx = [i for i in train_idx 
                     if events.iloc[i].t1 < t0_test or events.iloc[i].t0 > t1_test]
        
        # Apply embargo filter
        train_idx = [i for i in train_idx if i < test_start - embargo_size or i > test_end_embargo]
        
        yield np.array(train_idx), test_idx

for i, (train_idx, test_idx) in enumerate(purged_cv_split(labels, n_splits=10, embargo=0.05)):
    print(f"Fold {i+1}")
    print(" Train:", train_idx[:10])
    print(" Test:", test_idx[:10])
    print()



Fold 1
 Train: [150 151 152 153 154 155 156 157 158 159]
 Test: [0 1 2 3 4 5 6 7 8 9]

Fold 2
 Train: [0 1 2 3 4 5 6 7 8 9]
 Test: [100 101 102 103 104 105 106 107 108 109]

Fold 3
 Train: [0 1 2 3 4 5 6 7 8 9]
 Test: [200 201 202 203 204 205 206 207 208 209]

Fold 4
 Train: [0 1 2 3 4 5 6 7 8 9]
 Test: [300 301 302 303 304 305 306 307 308 309]

Fold 5
 Train: [0 1 2 3 4 5 6 7 8 9]
 Test: [400 401 402 403 404 405 406 407 408 409]

Fold 6
 Train: [0 1 2 3 4 5 6 7 8 9]
 Test: [500 501 502 503 504 505 506 507 508 509]

Fold 7
 Train: [0 1 2 3 4 5 6 7 8 9]
 Test: [600 601 602 603 604 605 606 607 608 609]

Fold 8
 Train: [0 1 2 3 4 5 6 7 8 9]
 Test: [700 701 702 703 704 705 706 707 708 709]

Fold 9
 Train: [0 1 2 3 4 5 6 7 8 9]
 Test: [800 801 802 803 804 805 806 807 808 809]

Fold 10
 Train: [0 1 2 3 4 5 6 7 8 9]
 Test: [900 901 902 903 904 905 906 907 908 909]

